In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

%matplotlib inline

In [ ]:
david = pd.read_csv(str(resolve('inputs/david/interaction_clustering.csv')))
david.head()

In [ ]:
interactions = pd.read_csv(str(resolve('database/interaction_heights.csv')))

interactions.head()

In [ ]:
#expand list of lists into their own rows
def explode(df, lst_cols, fill_value=np.nan, preserve_index=False):
    # make sure `lst_cols` is list-alike
    if (lst_cols is not None
        and len(lst_cols) > 0
        and not isinstance(lst_cols, (list, tuple, np.ndarray, pd.Series))):
        lst_cols = [lst_cols]
    # all columns except `lst_cols`
    idx_cols = df.columns.difference(lst_cols)
    # calculate lengths of lists
    lens = df[lst_cols[0]].str.len()
    # preserve original index values    
    idx = np.repeat(df.index.values, lens)
    # create "exploded" DF
    res = (pd.DataFrame({
                col:np.repeat(df[col].values, lens)
                for col in idx_cols},
                index=idx)
             .assign(**{col:np.concatenate(df.loc[lens>0, col].values)
                            for col in lst_cols}))
    # append those rows that have empty lists
    if (lens == 0).any():
        # at least one list in cells is empty
        res = (res.append(df.loc[lens==0, idx_cols], sort=False)
                  .fillna(fill_value))
    # revert the original index order
    res = res.sort_index()
    # reset index if requested
    if not preserve_index:        
        res = res.reset_index(drop=True)
    return res

In [ ]:
david.loc[:,'Genes'] = david['Genes'].apply(lambda x: x.split(','))
david_exp = explode(david, ['Genes'])
david_exp.loc[:,'Genes'] = david_exp['Genes'].apply(lambda x: x.strip())
#david_exp = david_exp[~(david_exp.GOTERM_MF_DIRECT ==  '')].copy()
#david_exp.loc[:,'GOTERM_MF_DIRECT'] = david_exp['GOTERM_MF_DIRECT'].apply(lambda x: x.split('~'))

In [ ]:
david_int = pd.merge(david_exp, interactions, left_on = 'Genes', right_on = 'Human ID link_prot1')
david_int = pd.merge(david_exp, david_int, left_on = 'Genes', right_on = 'Human ID link_prot2',
      suffixes = ('_prot1', '_prot2'))

david_int.head()

In [ ]:
david_same = david_int.loc[david_int['Annotation Cluster_prot1'] == david_int['Annotation Cluster_prot2']].copy()

In [ ]:
david_pivot = david_same[['Human interaction_id', 'Cluster Name_prot1', 'interaction_dim']].sort_values('interaction_dim').drop_duplicates(subset = ['Human interaction_id']).pivot('Cluster Name_prot1','Human interaction_id', 'interaction_dim')

In [ ]:
cmap = sns.cubehelix_palette(20, start=.5, rot=-.75)
sns.clustermap(david_pivot.fillna(0), cmap = cmap)

In [ ]:
#assign one functional annotation to each interaction: the highest combined fold enrichment, then the lowest
#combined FDR, then the cluster with the highest enrichment score, then the row carrying a CellphoneDB record.
#Any tie left is broken on the other columns in turn, and whole rows are kept, so the choice does not depend
#on row order: before, an unstable sort and groupby().first() let one added upstream row change 75 choices.

david_same.loc[:,'combined_FDR'] = david_same.loc[:,'FDR_prot1']+ david_same.loc[:,'FDR_prot2']
david_same.loc[:,'combined_fold_enrichment'] = david_same.loc[:,'Fold Enrichment_prot1']+ david_same.loc[:,'Fold Enrichment_prot2']

first = ['combined_fold_enrichment', 'combined_FDR', 'Enrichment Score_prot1', 'id_cp_interaction']
rest = [col for col in david_same.columns if col not in first]
david_unique = (david_same.sort_values(first + rest, ascending=[False, True, False, True] + [True] * len(rest), kind='mergesort')
                .drop_duplicates(subset = ['Human interaction_id'])
                .set_index('Human interaction_id').sort_index().reset_index())

In [ ]:
david_ufilter = david_unique[~((david_unique['Fold Enrichment_prot1'] <1)|(david_unique['Fold Enrichment_prot2'] <1))]

In [ ]:
sns.set_style('whitegrid')
sns.set_context("talk")
david_data = david_ufilter.drop_duplicates(subset = ['Human interaction_id','Cluster Name_prot1', 'Cluster Name_prot2'])
order_10 = pd.value_counts(david_data['Cluster Name_prot1']).iloc[:8].index

f, ax = plt.subplots(figsize=(8, 4))

sns.stripplot(x="interaction_dim", y="Cluster Name_prot1", order = order_10,
            data=david_data, color=(0.089,0.101,0.111),
            dodge=True, alpha=.25, zorder=1)

sns.pointplot(x="interaction_dim", y="Cluster Name_prot1", order = order_10,
            data= david_data, dodge = False, join=False, color=(0.801,0.28125,0.601),
              markers="d", scale=0.5, ci= None)



plt.xscale('log')

f.savefig(out_path('figures/F5_david_clusters.pdf'))

In [ ]:
group_nums = david_data.groupby('Cluster Name_prot1')['Human interaction_id'].count().sort_values(ascending = False).reset_index()

In [ ]:
f, ax = plt.subplots(figsize=(1, 4))
sns.barplot(x='Human interaction_id', y="Cluster Name_prot1", color=(0.089,0.101,0.111),
            data=group_nums[:8], alpha = 0.5)

f.savefig(out_path('figures/F5_david_cluster_counts.pdf'))

In [ ]:
david_data = david_unique.drop_duplicates(subset = ['Human interaction_id','Cluster Name_prot1', 'Cluster Name_prot2'])

sns.boxplot(x="interaction_dim", y="Cluster Name_prot1", order=pd.value_counts(david_data['Cluster Name_prot1']).iloc[:10].index,
            data=david_data)

plt.xscale('log')

In [ ]:
df = david_unique.drop_duplicates(subset = ['Human interaction_id', 'Cluster Name_prot1', 'Cluster Name_prot2'])

fig = plt.figure(figsize=(10,8), constrained_layout=True)
gs = fig.add_gridspec(9, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])


x = df['interaction_dim']
y = df.loc[(df['Cluster Name_prot1'] == 'tight junction')]['interaction_dim']
z = df.loc[(df['Cluster Name_prot1'] == 'Immunity')|(df['Cluster Name_prot1'] == 'Antigen Binding/Presentation')|(df['Cluster Name_prot1'] == 'Apoptosis/Inflammation')]['interaction_dim']
a = df.loc[(df['Cluster Name_prot1'] == 'Protein Tyrosine Kinase')|(df['Cluster Name_prot1'] == 'Ephrin Receptor')]['interaction_dim']
b = df.loc[(df['Cluster Name_prot1'] == 'sema/plexin')]['interaction_dim']
c = df.loc[(df['Cluster Name_prot1'] == 'GPI')]['interaction_dim']
d = df.loc[(df['Cluster Name_prot1'] == 'Integrin Adhesion')]['interaction_dim']
e = df.loc[(df['Cluster Name_prot1'] == 'Adherens Junction/Homophillic cell adhesion')]['interaction_dim']
f = df.loc[(df['Cluster Name_prot1'] == 'Notch')]['interaction_dim']
#d = df.loc[(df['Cluster Name_prot1'] >= 'gap junction(PC00105)')]['interaction_dim']
#e = df.loc[well & (df['Cluster Nameprot1'] == 6)]['interaction_dim']
#f = df.loc[well & (df['Cluster Nameprot1'] > 6)]['interaction_dim']


ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')  # rug plot
#ax1.set_ylim([-0.01, 0.01])
#ax1= plt.hist(x, bins=logbins, histtype = 'step', color = '#00A8A8') #histogram
#hist, bins, _ = plt.hist(x, bins=100)
#plt.xscale('log')
plt.xlabel('height (nm)')
#plt.ylabel('log count')
bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))


ax2 = fig.add_subplot(gs[1:2, 0], sharex = ax1)
plt.hist(y, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.plot(y, np.zeros(y.shape), 'b|', ms=20, color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax2.get_xticklabels(), visible=False)

ax3 = fig.add_subplot(gs[2:3, 0], sharex = ax1, sharey = ax2)
plt.plot(z, np.zeros(z.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(z, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
plt.ylabel('count')
plt.setp(ax3.get_xticklabels(), visible=False)

ax4 = fig.add_subplot(gs[3:4, 0], sharex = ax1, sharey = ax2)
plt.plot(a, np.zeros(a.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(a, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax4.get_xticklabels(), visible=False)

ax5 = fig.add_subplot(gs[4:5, 0], sharex = ax1, sharey = ax2)
plt.plot(b, np.zeros(b.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(b, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax5.get_xticklabels(), visible=False)

ax6 = fig.add_subplot(gs[5:6, 0], sharex = ax1, sharey = ax2)
plt.plot(c, np.zeros(c.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(c, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.setp(ax6.get_xticklabels(), visible=False)
plt.ylabel('count')

ax7 = fig.add_subplot(gs[6:7, 0], sharex = ax1, sharey = ax2)
plt.plot(d, np.zeros(d.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(d, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.setp(ax7.get_xticklabels(), visible=False)
plt.ylabel('count')

ax8 = fig.add_subplot(gs[7:8, 0], sharex = ax1, sharey = ax2)
plt.plot(e, np.zeros(e.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(e, bins=logbins, histtype = 'step', color = '#00A8A8')
#plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.setp(ax8.get_xticklabels(), visible=False)
plt.ylabel('count')

ax9 = fig.add_subplot(gs[8:9, 0], sharex = ax1, sharey = ax2)
plt.plot(f, np.zeros(f.shape), 'b|', ms=20, color = '#00A8A8')
plt.hist(f, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
plt.xlabel('log height (nm)')
plt.ylabel('count')

In [ ]:
df.to_csv(out_path('database/david_interaction_clustering_long.csv'))